# Feature engineering – Délinquance
**Contributeurs :**Cécile & Théophile · **Date :** 26/10/2026

Maille : commune × année. La cible n'est pas utilisée ici (déjà étudiée dans l'EDA).

## 0. Chargement

In [67]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# config.py se trouve à la racine du projet, dossier parent du notebook
sys.path.insert(0, str(Path.cwd().parent))
from config import DATA_OUT

CLES = ["code_commune", "annee"]    # clés de jointure avec la table cible

df = pd.read_parquet(DATA_OUT / "delinquance_commune_annee.parquet")

# Contrôle : une seule ligne par commune et par année
print(df.shape)
print("Doublons sur les clés :", df.duplicated(CLES).sum())
print("Années :", sorted(df["annee"].unique()))
df.columns.tolist()

(349170, 35)
Doublons sur les clés : 0
Années : [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


['code_commune',
 'annee',
 'nb_violences_physiques_intrafamiliales',
 'nb_violences_physiques_hors_cadre_familial',
 'nb_violences_sexuelles',
 'nb_vols_avec_armes',
 'nb_vols_violents_sans_arme',
 'nb_vols_sans_violence_contre_des_personnes',
 'nb_cambriolages_de_logement',
 'nb_vols_de_vehicule',
 'nb_vols_dans_les_vehicules',
 'nb_vols_d_accessoires_sur_vehicules',
 'nb_destructions_et_degradations_volontaires',
 'nb_usage_de_stupefiants',
 'nb_usage_de_stupefiants_afd',
 'nb_trafic_de_stupefiants',
 'nb_escroqueries_et_fraudes_aux_moyens_de_paiement',
 'insee_pop',
 'insee_log',
 'n_indicateurs_estimes',
 'taux_violences_physiques_intrafamiliales',
 'taux_violences_physiques_hors_cadre_familial',
 'taux_violences_sexuelles',
 'taux_vols_avec_armes',
 'taux_vols_violents_sans_arme',
 'taux_vols_sans_violence_contre_des_personnes',
 'taux_cambriolages_de_logement',
 'taux_vols_de_vehicule',
 'taux_vols_dans_les_vehicules',
 'taux_vols_d_accessoires_sur_vehicules',
 'taux_destruction

## 1. Constats de l'EDA utilisés

L'EDA a été faite au bassin de vie. Les constats sont repris ici pour un travail à la commune, où les taux sont moins stables (moins de faits par commune). La cible retenue est le niveau du prix : ce sont les liens avec le niveau (C1) qui guident la sélection.

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| Vols de véhicule, vols dans les véhicules | C1 : vols dans les véhicules 0,50 (maisons), le lien le plus fort. B2 : famille la plus territorialisée (η² = 0,49) | Regroupés (même unité : véhicule) |
| Vols d'accessoires sur véhicules | C1 : pas de lien avec le prix (0,08), dilue sa famille | Exclus |
| Vols avec armes, vols violents sans arme | C1 : la famille (0,44) dépasse ses deux indicateurs. B2 : faits très rares (médiane 0,17 pour 1 000) | Regroupés (même unité : infraction) |
| Vols sans violence | C1 : 0,34. B4 : l'indicateur le plus stable (rho = 0,88) | Gardés |
| Cambriolages | B1 : la famille la plus à part. B2 : très territorialisés (η² = 0,42). C1 : lien faible | Gardés (pour 1 000 logements) |
| Destructions et dégradations | C1 : lien faible. B1 : rho = 0,77 avec les vols sans violence | Gardées (redondance à recontrôler à la commune) |
| Usage et trafic de stupéfiants | C1 : lien faible (0,21). B3 : reflètent en partie l'activité des services | Regroupés (même unité : mis en cause) |
| Usage de stupéfiants (AFD) | Déjà compris dans l'usage ; nul avant 2021 | Exclu |
| Violences physiques intrafamiliales | C1 : −0,25 et −0,24. Lien probablement dû aux caractéristiques sociales du territoire | Gardées seules, apport à vérifier en modélisation |
| Violences sexuelles | C1 : −0,36 et −0,30, le lien négatif le plus net | Gardées seules : faits de nature différente des violences physiques, contexte (famille ou extérieur) inconnu |
| Violences hors cadre familial | C1 : pas de lien avec le prix (0,02 et −0,10) | Exclues |
| Escroqueries | B2 : la famille la plus homogène (D9 / D1 = 1,7). B4 : 36 % des écarts entre bassins. C1 : lien faible | Exclues |
| Tous les taux | B3 et B4 : caractéristique stable du territoire ; évolutions nationales dues à des facteurs extérieurs | Pas de variable d'évolution |
| Tous les taux | Distributions asymétriques, valeurs extrêmes (audit, section 7) | À décider : transformation |
| Secret statistique | Faits masqués remplacés par la moyenne départementale (bloc 8) | Indicateur de fiabilité : part des 15 indicateurs estimés |
| Petits territoires | Taux instables quand il y a peu de faits (B3.4 : vols violents, rho = 0,59) | Taux lissé sur 3 ans, calculé après la fusion à la maille choisie (bassin ou sous-bassin) |

## 2. Sélection et regroupement des variables

### 2.1 Regroupement en familles
On additionne les nombres de faits des indicateurs de même nature et de même unité de compte. Aucun taux n'est calculé ici : les taux seront calculés après la fusion avec la table de la maille de modélisation, à partir de la population et de la superficie du territoire. La transformation des variables asymétriques et la mise à l'échelle se feront à ce moment-là.

In [68]:
# Indicateurs gardés, regroupés par famille (même nature, même unité de compte)
FAMILLES = {
    "vols_vehicules":            ["nb_vols_de_vehicule", "nb_vols_dans_les_vehicules"],
    "vols_violents":             ["nb_vols_avec_armes", "nb_vols_violents_sans_arme"],
    "vols_sans_violence":        ["nb_vols_sans_violence_contre_des_personnes"],
    "cambriolages":              ["nb_cambriolages_de_logement"],
    "destructions":              ["nb_destructions_et_degradations_volontaires"],
    "stupefiants":               ["nb_usage_de_stupefiants", "nb_trafic_de_stupefiants"],
    "violences_intrafamiliales": ["nb_violences_physiques_intrafamiliales"],
    "violences_sexuelles":       ["nb_violences_sexuelles"],
}

# Dénominateurs du SSMSI gardés pour calculer les taux après la fusion
fe = df[CLES + ["insee_pop", "insee_log"]].copy()
for famille, colonnes in FAMILLES.items():
    # min_count=1 : la somme reste vide si tous les indicateurs de la famille sont vides
    fe["nb_" + famille] = df[colonnes].sum(axis=1, min_count=1)

print(fe.shape)
print("Valeurs vides par famille :")
print(fe.filter(like="nb_").isna().sum())

(349170, 12)
Valeurs vides par famille :
nb_vols_vehicules               0
nb_vols_violents                0
nb_vols_sans_violence           0
nb_cambriolages                 0
nb_destructions                 0
nb_stupefiants                  0
nb_violences_intrafamiliales    0
nb_violences_sexuelles          0
dtype: int64


### 2.2 Décalage d'un an
Les faits de l'année N servent à expliquer les prix de l'année N+1 : les chiffres d'une année ne sont arrêtés qu'en avril de l'année suivante. La colonne `annee` devient l'année des ventes associées, comme pour LOVAC ; l'année des faits est gardée dans `annee_faits`.

In [69]:
# annee = année des ventes que ces faits servent à expliquer (faits de N -> prix de N+1)
# L'année est lue dans df, qui n'est jamais modifié : la cellule peut être relancée sans risque
fe["annee_faits"] = df["annee"]
fe["annee"] = df["annee"] + 1

print(fe[["annee_faits", "annee"]].drop_duplicates().sort_values("annee").to_string(index=False))

 annee_faits  annee
        2016   2017
        2017   2018
        2018   2019
        2019   2020
        2020   2021
        2021   2022
        2022   2023
        2023   2024
        2024   2025
        2025   2026


### 2.3 Fiabilité
Part des 15 indicateurs estimés à cause du secret statistique : elle sert à repérer les territoires dont les taux sont peu fiables.

In [70]:
# Fiabilité : part des 15 indicateurs estimés à cause du secret statistique
fe["part_indicateurs_estimes"] = df["n_indicateurs_estimes"] / 15 * 100
print("Part des indicateurs estimés (commune) : moyenne =", round(fe["part_indicateurs_estimes"].mean(), 1), "%")

Part des indicateurs estimés (commune) : moyenne = 46.3 %


**Résultat :** en moyenne, 46 % des indicateurs d'une commune sont estimés.

## 3. Variables qualitatives
Aucune variable qualitative. Les valeurs masquées par le secret statistique ont été remplacées par le complément départemental dans la préparation (bloc 8) ; elles sont signalées par la part des indicateurs estimés (section 2.3). Les nombres de faits des communes masquées sont donc des estimations, pas toujours entières.

## 4. Tests statistiques
Reportés après la fusion : ils doivent porter sur les taux. Les nombres de faits suivent la taille de la commune et seraient tous fortement liés entre eux.

## 5. Variables retenues

In [72]:
CLES = ["code_commune", "annee"]   # annee = année des ventes associées
VARIABLES = ["annee_faits", "insee_pop", "insee_log","part_indicateurs_estimes"] + ["nb_" + f for f in FAMILLES]

df_fe = fe[CLES + VARIABLES]

# Contrôle des clés avant l'export
print(df_fe.shape)
print("Doublons sur les clés :", df_fe.duplicated(CLES).sum())

(349170, 14)
Doublons sur les clés : 0


## 6. Fusion des variables retenues avec la cible, export

La fusion avec la cible doit se faire en plusieurs étapes : d'abord la jonction entre le fichier après feature engineering pour avoir pour chaque communes les informations liées au dataset travaillé et le sous bassin de vie auquel chaque commune fait partie, puis la jonction avec la table cible via le sous bassin de vie et l'année des observations.

In [73]:
cible=pd.read_parquet(DATA_OUT / "cible_sbv_annee.parquet")
liaison=pd.read_parquet(DATA_OUT / "liaison_bv_scindes.parquet")

com_fe=pd.merge(liaison, df_fe, left_on="CODGEO_INI", right_on="code_commune").drop(columns='code_commune')
df_final=pd.merge(com_fe,cible, left_on=["sous_bassin_de_vie", "annee"], right_on=["sous_bassin_de_vie", "annee"])
print(df_final.columns)

df_final.to_parquet(DATA_OUT / "fe_cible_delinquance_commune.parquet", index=False)

Index(['CODGEO_INI', 'CODGEO_2026', 'BV2022', 'TYPE_BV2022',
       'sous_bassin_de_vie', 'pop_bv_totale', 'population_sbv',
       'superficie_sbv_km2', 'annee', 'annee_faits', 'insee_pop', 'insee_log',
       'part_indicateurs_estimes', 'nb_vols_vehicules', 'nb_vols_violents',
       'nb_vols_sans_violence', 'nb_cambriolages', 'nb_destructions',
       'nb_stupefiants', 'nb_violences_intrafamiliales',
       'nb_violences_sexuelles', 'n_appart', 'n_maison', 'prix_m2_med_appart',
       'prix_m2_med_maison'],
      dtype='str')


## 7. Après la fusion
- Contrôle de la qualité des données après la jointure.
- Calcul des taux : nombres et dénominateurs (`insee_pop`, `insee_log`) additionnés à la maille de modélisation, puis faits pour 1 000 habitants et par km² ; cambriolages pour 1 000 logements.
- Tests entre les taux (redondance vols sans violence / destructions), puis choix de la transformation des taux asymétriques.
- Mise à l'échelle juste avant la modélisation, avec des paramètres calculés sur les années d'apprentissage seulement.
- Taux lissés sur 3 ans (faits et dénominateurs des 3 dernières années additionnés), surtout utiles pour les petits territoires et les sous-bassins : fonction `taux_lisses` ci-dessous, à appeler avec la maille finale.

In [33]:
def taux_lisses(table, maille, familles, fenetre=3):
    """Taux annuels (taux1_) et lissés sur `fenetre` années (taux3_) par maille x année,
    pour 1 000 habitants (1 000 logements pour les cambriolages).
    table : fichier fe_delinquance_commune joint à la maille (colonne `maille`).
    Le taux lissé reste vide tant que la fenêtre n'est pas complète (premières années)."""
    table = table.copy()
    table["estimes_x_pop"] = table["part_indicateurs_estimes"] * table["insee_pop"]
    colonnes = ["nb_" + f for f in familles] + ["insee_pop", "insee_log", "estimes_x_pop"]
    # 1. On ADDITIONNE faits et dénominateurs par maille et par année
    agr = table.groupby([maille, "annee"])[colonnes].sum().reset_index()
    agr = agr.sort_values([maille, "annee"]).reset_index(drop=True)
    # 2. Fiabilité : moyenne pondérée par la population
    agr["part_indicateurs_estimes"] = agr["estimes_x_pop"] / agr["insee_pop"]
    # 3. Taux annuel et taux lissé (années passées uniquement, fenêtre complète)
    for f in familles:
        denom = "insee_log" if f == "cambriolages" else "insee_pop"
        faits_n = agr.groupby(maille)["nb_" + f].rolling(fenetre, min_periods=fenetre).sum().reset_index(level=0, drop=True)
        denom_n = agr.groupby(maille)[denom].rolling(fenetre, min_periods=fenetre).sum().reset_index(level=0, drop=True)
        agr["taux1_" + f] = agr["nb_" + f] / agr[denom] * 1000
        agr["taux3_" + f] = faits_n / denom_n * 1000
    return agr.drop(columns="estimes_x_pop")

# Feature engineering – LOVAC (logements privés vacants)
**Contributeurs :** Cécile & Théophile · **Date :** 26/10/2026

Maille : commune (photo fixe d'un millésime). La cible n'est pas utilisée ici (déjà étudiée dans l'EDA).

## 0. Chargement
Table produite par le bloc 9 de la préparation : une ligne par commune et par millésime (2020 à 2026). Le millésime N décrit la situation au 1er janvier N-1.

In [74]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from config import DATA_OUT

df = pd.read_parquet(DATA_OUT / "lovac_commune_annee.parquet")

# Contrôle : une seule ligne par commune et par millésime
print(df.shape)
print("Doublons :", df.duplicated(["code_commune", "millesime"]).sum())
print("Valeurs vides :")
print(df[["vacants", "vacants_2ans", "parc_prive"]].isna().groupby(df["millesime"]).sum())

(244384, 18)
Doublons : 0
Valeurs vides :
           vacants  vacants_2ans  parc_prive
millesime                                   
2020          7396         20278       18599
2021          7184         20030       18521
2022          7459         20343       18515
2023          7202         19826       18485
2024          6866         26927       18401
2025          9395         22135       18321
2026          6882         17767       34912


## 1. Constats de l'EDA utilisés

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| `taux_vacance_complete` | C4 : ρ = −0,44 (maisons), −0,33 (appartements). Parc privé caché pour environ 53 % des communes, surtout les petites (B2.7 : 85 % des communes de moins de 100 logements ; 90,6 % des logements sont dans une commune au parc connu) | Gardé ; taux caché complété par celui de l'EPCI (préparation, bloc 9) |
| `taux_impute` | Signale les taux complétés par l'EPCI | Gardé pour le contrôle qualité après la jointure |
| `taux_vacance_2ans` | C4 : ρ = −0,70 et −0,57, mais environ 85 % de valeurs manquantes, non complétées : lien mesuré sur les seules communes visibles, surtout les plus grandes | Écartée a priori. Variable à suivre pour les grandes agglomérations, quand elles seront séparées de leur bassin de vie |
| `part_longue_parmi_vacants` | C4 : doublon de `taux_vacance_2ans` (0,88) | Exclue |
| Millésime 2026 | Pas de parc privé : taux impossible | Hors apprentissage et test ; sert seulement à la simulation finale |
| Tous les millésimes | Classement des communes stable entre millésimes (B2.6 : ρ de 0,87 à 0,92) ; la cible est le niveau du prix | Une seule photo pour toutes les années, comme le recensement (plan 5.3) |
| Rupture de série | B2.6 : la chute nationale apparaît dans le millésime 2025 (situation 2024 : 7,2 % contre environ 11 %) | Photo du millésime 2023, avant la rupture ; elle couvre aussi les prix 2014-2019, sans LOVAC sinon |

## 2. Sélection des variables
On garde le taux de vacance complété par l'EPCI et l'indicateur de complétion, pour tous les millésimes ; la photo du millésime 2023 est extraite à la section 5.

In [75]:
CLES = ["code_commune", "annee"]   # annee = année des ventes associées

fe = df.rename(columns={"annee_vente_associee": "annee"})
fe = fe[CLES + ["millesime", "taux_vacance_complete", "taux_impute", "apres_rupture"]].copy()

# Contrôle : clés uniques, valeurs vides par année
print(fe.shape)
print("Doublons sur les clés :", fe.duplicated(CLES).sum())
print(fe.groupby("annee")[["taux_vacance_complete"]].apply(lambda g: g.isna().sum()))

(244384, 6)
Doublons sur les clés : 0
       taux_vacance_complete
annee                       
2020                       0
2021                       0
2022                       0
2023                       0
2024                       0
2025                       0
2026                   34912


## 3. Variables qualitatives
Aucune à encoder : `taux_impute` est déjà binaire (vrai / faux). `apres_rupture` n'est plus utile avec une photo fixe d'avant la rupture.

## 4. Tests statistiques
Reportés après la fusion : ils doivent porter sur les taux à la maille finale (redondance avec la vacance du recensement, part des logements dont le taux est complété par l'EPCI).

## 5. Variables retenues
Photo fixe : millésime 2023 (situation au 1er janvier 2022), recopiée pour toutes les années de vente à la fusion. La clé est donc `code_commune`, sans année.

In [76]:
MILLESIME_PHOTO = 2023      # avant la rupture de série (millésime 2025)

df_fe = fe.loc[fe["millesime"] == MILLESIME_PHOTO,
               ["code_commune", "millesime", "taux_vacance_complete", "taux_impute"]]

# Contrôles : une ligne par commune, pas de taux vide
print(df_fe.shape)
print("Doublons :", df_fe["code_commune"].duplicated().sum(),
      "| taux vides :", df_fe["taux_vacance_complete"].isna().sum())

(34912, 4)
Doublons : 0 | taux vides : 0


## 6. Jointure avec la cible et export

La fusion avec la cible doit se faire en plusieurs étapes : d'abord la jonction entre le fichier après feature engineering pour avoir pour chaque communes les informations liées au dataset travaillé et le sous bassin de vie auquel chaque commune fait partie, puis la jonction avec la table cible via le sous bassin de vie et l'année des observations.

In [77]:
cible=pd.read_parquet(DATA_OUT / "cible_sbv_annee.parquet")
liaison=pd.read_parquet(DATA_OUT / "liaison_bv_scindes.parquet")

print(df_fe.columns)

com_fe=pd.merge(liaison, df_fe, left_on="CODGEO_INI", right_on="code_commune").drop(columns='code_commune')
df_final=pd.merge(com_fe,cible, left_on=["sous_bassin_de_vie"], right_on=["sous_bassin_de_vie"])
print(df_final.columns)

df_final.to_parquet(DATA_OUT / "fe_cible_vacance_commune.parquet", index=False)

Index(['code_commune', 'millesime', 'taux_vacance_complete', 'taux_impute'], dtype='str')
Index(['CODGEO_INI', 'CODGEO_2026', 'BV2022', 'TYPE_BV2022',
       'sous_bassin_de_vie', 'pop_bv_totale', 'population_sbv',
       'superficie_sbv_km2', 'millesime', 'taux_vacance_complete',
       'taux_impute', 'annee', 'n_appart', 'n_maison', 'prix_m2_med_appart',
       'prix_m2_med_maison'],
      dtype='str')


## 7. Après la fusion
- Contrôle qualité : part des communes dont le taux a été complété par l'EPCI, par territoire.
- Passage au territoire : moyenne des taux des communes, pondérée par le nombre de logements du recensement (`P22_LOG`). Le parc privé est inconnu pour les communes masquées ; les logements du recensement comprennent le parc social (approximation).
- Clé : `code_commune`, sans année (LOVAC 2023 fixe pour toutes les années de vente).
- Transformation éventuelle et mise à l'échelle juste avant la modélisation (paramètres calculés sur les années d'apprentissage seulement).

# Feature engineering – Équipements (BPE en évolution)
**Contributeurs :** Cécile & Théophile · **Date :** 27/10/2026

Maille : bassin de vie (la source n'existe pas à la commune). La cible n'est pas utilisée ici (déjà étudiée dans l'EDA complémentaire).

## 0. Chargement

In [78]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from config import DATA_RAW, DATA_OUT

fichier = DATA_RAW / "DS_BPE_EVOLUTION_2025_data.csv"
separateur = ";" if ";" in fichier.open(encoding="utf-8").readline() else ","   # séparateur lu sur la 1re ligne
df = pd.read_csv(fichier, sep=separateur, dtype=str)

# BPE 2020 au bassin de vie, métropole seulement (outre-mer hors périmètre de DVF+)
bpe20 = df[(df["GEO_OBJECT"] == "BV2022") & (df["TIME_PERIOD"] == "2020") & ~df["GEO"].str.startswith("97")].copy()
bpe20["OBS_VALUE"] = pd.to_numeric(bpe20["OBS_VALUE"])
print(bpe20.shape, "| bassins :", bpe20["GEO"].nunique())

(157298, 9) | bassins : 1681


## 1. Constats de l'EDA utilisés

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| Source | Audit : BPE en évolution, nombres d'équipements comparables en 2015, 2020 et 2025, directement au bassin de vie | Retenue à la place de la table de présence par commune |
| Enseignement | Audit : 3 types, équipements de l'Éducation nationale absents | Exclu |
| Six autres domaines | B1 : rho entre domaines ≤ 0,64, pas de redondance. C1 ter : tous liés au prix à densité égale (rho partiels de 0,22 à 0,57 ; sports −0,26) | Gardés, sports compris |
| Millésime | B3 et B4 : rho de 0,82 à 0,99 entre millésimes ; 83 à 99 % des écarts viennent des différences entre bassins. Interpolation avec 2025 = information publiée après la date de prédiction | BPE 2020 fixe pour toutes les années, pas de variable d'évolution |
| Forme | C1 : le taux par habitant mesure aussi le caractère rural. C1 bis : les équipements par km² suivent la densité (rho de 0,81 à 0,96) | Taux pour 10 000 habitants + densité de population, calculés après la fusion |

## 2. Sélection et création des variables

### 2.1 Nombre d'équipements par domaine
Nombre d'équipements de 2020 par bassin de vie et par domaine (première lettre du code d'équipement), tous les types renseignés en 2020. Les taux seront calculés après la fusion, avec la population de la maille de modélisation.

In [79]:
DOMAINES = {"A": "services", "B": "commerces", "D": "sante",
            "E": "transports", "F": "sports", "G": "tourisme"}   # C (enseignement) exclu

bpe20["domaine"] = bpe20["FACILITY_TYPE"].str[0].map(DOMAINES)
equip = (bpe20.dropna(subset=["domaine"])
              .pivot_table(index="GEO", columns="domaine", values="OBS_VALUE", aggfunc="sum")
              .add_prefix("nb_equip_")
              .reset_index()
              .rename(columns={"GEO": "BV2022"}))
equip.columns.name = None

print(equip.shape)
print("Valeurs vides :", equip.isna().sum().sum())
equip.describe().round(0)

(1681, 7)
Valeurs vides : 40


,nb_equip_commerces,nb_equip_sante,nb_equip_services,nb_equip_sports,nb_equip_tourisme,nb_equip_transports
count,1681.0,1681.0,1681.0,1681.0,1662.0,1660.0
mean,217.0,207.0,638.0,112.0,25.0,45.0
std,1590.0,1434.0,4352.0,356.0,189.0,1017.0
min,12.0,9.0,47.0,8.0,0.0,0.0
25%,39.0,36.0,162.0,39.0,4.0,3.0
50%,69.0,60.0,263.0,64.0,9.0,7.0
75%,140.0,116.0,447.0,108.0,22.0,13.0
max,62405.0,54140.0,170344.0,13290.0,7516.0,41286.0


### 2.2 Superficie des communes
Nécessaire pour la densité de population (constat C1 bis) et pour les taux par km² des autres sources. Gardée à la commune (géographie 2026) : elle s'additionne à n'importe quelle maille, y compris les futures sous-unités de bassin. Source : référentiel des communes 2026 (data.gouv.fr, Insee et IGN). Paris, Lyon et Marseille n'y figurent que comme villes entières.

In [80]:
superficie = pd.read_csv(DATA_RAW / "communes-france-2026.csv", usecols=["code_insee", "superficie_km2"],
                         dtype={"code_insee": str})
superficie = superficie.rename(columns={"code_insee": "code_commune_2026"})

print(superficie.shape)
print("Doublons :", superficie.duplicated("code_commune_2026").sum(),
      "| superficies vides :", superficie["superficie_km2"].isna().sum())

(34868, 2)
Doublons : 0 | superficies vides : 129


## 3. Variables qualitatives
Aucune.

## 4. Tests statistiques
Faits dans l'EDA complémentaire (B1 à C1 ter) ; à refaire sur les taux après la fusion si la maille change (sous-unités de bassin).

### 4.1 Case vide = aucun équipement, ou information manquante ?
Après le pivot, 40 cases sont vides (tourisme et transports). On vérifie comment l'Insee construit le fichier, puis si ces cases concernent des petits bassins.


In [81]:
from scipy import stats

bv = df[(df["GEO_OBJECT"] == "BV2022") & ~df["GEO"].str.startswith("97")].copy()
bv["OBS_VALUE"] = pd.to_numeric(bv["OBS_VALUE"])

# a) Une ligne (bassin, type) est-elle écrite pour les 3 années à la fois ?
lignes = bv.groupby(["GEO", "FACILITY_TYPE"])["TIME_PERIOD"].nunique()
print("Nombre d'années par couple (bassin, type) :", lignes.value_counts().to_dict())
print("Couples écrits :", len(lignes), "sur", bv["GEO"].nunique() * bv["FACILITY_TYPE"].nunique(), "possibles")

# b) Les 0 de 2020 sont-ils des types présents une autre année ?
v2020 = bv[bv["TIME_PERIOD"] == "2020"].set_index(["GEO", "FACILITY_TYPE"])["OBS_VALUE"]
autres = bv[bv["TIME_PERIOD"] != "2020"].groupby(["GEO", "FACILITY_TYPE"])["OBS_VALUE"].max()
zeros = v2020[v2020 == 0]
print("Zéros en 2020 :", len(zeros), "| dont présents en 2015 ou 2025 :",
      round((autres.reindex(zeros.index) > 0).mean() * 100, 1), "%")

# c) Les bassins avec une case vide sont-ils plus petits ? (Welch sur le log des logements, d de Cohen)
logement = pd.read_parquet(DATA_OUT / "logement_commune_2022.parquet", columns=["code_commune", "P22_LOG"])
liaison = pd.read_parquet(DATA_OUT / "liaison_commune_bv2022.parquet", columns=["CODGEO_INI", "BV2022"])
log_bv = logement.merge(liaison, left_on="code_commune", right_on="CODGEO_INI", how="inner")
log_bv = log_bv.groupby("BV2022")["P22_LOG"].sum().reset_index()
t = equip.merge(log_bv, on="BV2022", how="left")
t["log_logements"] = np.log10(t["P22_LOG"])
for domaine in ["nb_equip_tourisme", "nb_equip_transports"]:
    vide = t[domaine].isna()
    a = t.loc[vide, "log_logements"]
    b = t.loc[~vide, "log_logements"]
    t_stat, p = stats.ttest_ind(a, b, equal_var=False)
    d = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
    print(domaine, ": bassins vides =", vide.sum(),
          "| logements médians vides / autres =", round(t.loc[vide, "P22_LOG"].median()), "/", round(t.loc[~vide, "P22_LOG"].median()),
          "| t =", round(t_stat, 1), "| p-value =", "%.1e" % p, "| d =", round(d, 2))

# Conclusion : une case vide = aucun équipement de ce domaine dans le bassin
equip = equip.fillna(0)
print("Cases vides restantes :", equip.isna().sum().sum())

Nombre d'années par couple (bassin, type) : {3: 157298}
Couples écrits : 157298 sur 255512 possibles
Zéros en 2020 : 11581 | dont présents en 2015 ou 2025 : 100.0 %
nb_equip_tourisme : bassins vides = 19 | logements médians vides / autres = 4215 / 9093 | t = -14.9 | p-value = 1.9e-13 | d = -1.41
nb_equip_transports : bassins vides = 21 | logements médians vides / autres = 3801 / 9093 | t = -10.6 | p-value = 3.0e-10 | d = -1.36
Cases vides restantes : 0


**Constat :**
- L'Insee n'écrit une ligne (bassin, type) que si le type a été présent au moins une des 3 années, et l'écrit alors pour les 3 années. Les 11 581 zéros de 2020 sont tous des types présents en 2015 ou en 2025.
- Les 40 cases vides sont des petits bassins : 4 215 (tourisme) et 3 801 (transports) logements médians, contre 9 093 pour les autres (Welch p < 0,001, d = −1,41 et −1,36, effet fort).
- **Une case vide veut donc dire « aucun équipement de ce domaine » : elle est remplacée par 0.** Le seul vrai « inconnu », le type D304 (statut O dans 146 bassins), est déjà ignoré par la somme.

## 5. Variables retenues

In [82]:
print("Équipements :", equip.shape, "| doublons :", equip.duplicated("BV2022").sum())

Équipements : (1681, 7) | doublons : 0


## 6. Fusion des variables retenues avec la cible, export

La fusion avec la cible doit se faire en plusieurs étapes : d'abord la jonction entre le fichier après feature engineering pour avoir pour chaque bassin de vie les informations liées au dataset travaillé et le sous bassin de vie auquel chaque commune fait partie, puis la jonction avec la table cible via le sous bassin de vie et l'année des observations.

In [83]:
cible=pd.read_parquet(DATA_OUT / "cible_sbv_annee.parquet")
liaison=pd.read_parquet(DATA_OUT / "liaison_bv_scindes.parquet")

com_fe=pd.merge(equip, liaison, left_on="BV2022", right_on="BV2022")
df_final=pd.merge(com_fe,cible, left_on=["sous_bassin_de_vie"], right_on=["sous_bassin_de_vie"])
df_final=df_final.drop(columns=['CODGEO_INI', 'CODGEO_2026'])
print(df_final.columns)

df_final.to_parquet(DATA_OUT / "fe_cible_equipement_bv.parquet", index=False)

Index(['BV2022', 'nb_equip_commerces', 'nb_equip_sante', 'nb_equip_services',
       'nb_equip_sports', 'nb_equip_tourisme', 'nb_equip_transports',
       'TYPE_BV2022', 'sous_bassin_de_vie', 'pop_bv_totale', 'population_sbv',
       'superficie_sbv_km2', 'annee', 'n_appart', 'n_maison',
       'prix_m2_med_appart', 'prix_m2_med_maison'],
      dtype='str')


## 7. Après la fusion
- Clé : `BV2022`, sans année (BPE 2020 fixe pour toutes les années de vente).
- Densité de population : population municipale 2020 (`insee_pop`, fichier de délinquance) additionnée par bassin, divisée par la superficie additionnée par bassin (`fe_superficie_commune`, rattachée par `CODGEO_2026` de la table de liaison).
- Taux : nombre d'équipements par domaine pour 10 000 habitants (population 2020).
- Sous-unités de bassin (ville-centre, reste) : la BPE en évolution n'existe pas à la commune ; nombres à reconstruire avec la BPE 2025 communale (concordance de 99 % au bassin), en gardant à l'esprit qu'elle décrit 2025.
- Transformation éventuelle et mise à l'échelle juste avant la modélisation (paramètres calculés sur les années d'apprentissage seulement).

# Feature engineering – Parc de logements (recensement)
**Contributeurs :** Cécile & Théophile · **Date :** 27/10/2026

Maille : commune. La cible n'est pas utilisée ici (déjà étudiée dans l'EDA et l'EDA complémentaire).

## 0. Chargement
Table produite par le bloc 7 de la préparation : une ligne par commune (Paris, Lyon, Marseille par arrondissement), recensement 2022, effectifs bruts et indicateurs.

In [84]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from config import DATA_OUT

df = pd.read_parquet(DATA_OUT / "logement_commune_2022.parquet")

print(df.shape)
print("Doublons :", df.duplicated("code_commune").sum())

(34955, 52)
Doublons : 0


## 1. Constats de l'EDA utilisés

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| Sélection | Rapport, section 4.8 : 12 indicateurs gardés (\|rho\| > 0,2 à la commune, doublons > 0,8 écartés) | Les 12 gardés |
| Millésime | EDA complémentaire : de 2016 à 2022, le choix du millésime fait varier le rho avec le prix de 0,06 au plus ; les bassins gardent leur classement | Photo 2022 pour toutes les années, pas de variable d'évolution |
| Taux de vacance | EDA complémentaire : rho de −0,76 au bassin de vie (maisons et appartements) | Gardé, à côté du taux de LOVAC (redondance à contrôler après la fusion) |
| Taille des ménages | Même définition que la taille moyenne des ménages du bloc 6 | Gardée ; une seule des deux à retenir après la fusion |
| Effectifs | Au bassin de vie, les parts se recalculent en additionnant les effectifs | Effectifs bruts exportés, indicateurs calculés après la fusion |

## 2. Sélection des variables
Pour chacun des 12 indicateurs, on garde les effectifs du numérateur et du dénominateur.

In [85]:
# Effectifs nécessaires à chaque indicateur retenu (numérateur puis dénominateur)
INDICATEURS = {
    "taux_vacance":       ["P22_LOGVAC", "P22_LOG"],
    "part_rsec":          ["P22_RSECOCC", "P22_LOG"],
    "part_apparts":       ["P22_APPART", "P22_LOG"],
    "log_nb_logements":   ["P22_LOG"],
    "part_moins40m2":     ["P22_RP_M30M2", "P22_RP_3040M2", "P22_RP"],
    "part_avant1946":     ["P22_RP_ACH1919", "P22_RP_ACH1945", "P22_RP"],
    "part_2006_2019":     ["P22_RP_ACH2019", "P22_RP"],
    "anciennete_moyenne": ["P22_ANEM_RP", "P22_RP"],
    "taille_menage":      ["P22_NPER_RP", "P22_RP"],
    "part_sans_voiture":  ["P22_RP_VOIT1P", "P22_MEN"],
    "part_parking":       ["P22_RP_GARL", "P22_MEN"],
    "part_suroccupes":    ["C22_RP_SUROCC_MOD", "C22_RP_SUROCC_ACC", "P22_RP"],
}
EFFECTIFS = sorted({c for cols in INDICATEURS.values() for c in cols})

fe = df[["code_commune", "annee_rp"] + EFFECTIFS].copy()
print(fe.shape)
print("Valeurs vides :", fe[EFFECTIFS].isna().sum().sum())

(34955, 19)
Valeurs vides : 0


## 3. Variables qualitatives
Aucune. Les colonnes d'aide de la préparation (Alsace-Moselle, petites communes) ne sont pas reprises : l'Alsace-Moselle sort à la jointure avec la cible, et le seuil de 20 ventes s'applique à la maille de modélisation.

## 4. Tests statistiques
Faits dans le rapport (section 4.8, à la commune) et dans l'EDA complémentaire (au bassin de vie) ; à refaire sur les indicateurs après la fusion.

## 5. Variables retenues et export

In [86]:
print(fe.shape, "| doublons :", fe.duplicated("code_commune").sum())

(34955, 19) | doublons : 0


## 6. Fusion des variables retenues avec la cible, export

La fusion avec la cible doit se faire en plusieurs étapes : d'abord la jonction entre le fichier après feature engineering pour avoir pour chaque communes les informations liées au dataset travaillé et le sous bassin de vie auquel chaque commune fait partie, puis la jonction avec la table cible via le sous bassin de vie et l'année des observations.

In [87]:
cible=pd.read_parquet(DATA_OUT / "cible_sbv_annee.parquet")
liaison=pd.read_parquet(DATA_OUT / "liaison_bv_scindes.parquet")

com_fe=pd.merge(fe, liaison, left_on="code_commune", right_on="CODGEO_INI").drop(columns='code_commune')
df_final=pd.merge(com_fe,cible, left_on=["sous_bassin_de_vie", "annee_rp"], right_on=["sous_bassin_de_vie", "annee"])
print(df_final.columns)

df_final.to_parquet(DATA_OUT / "fe_cible_logement_commune.parquet", index=False)

Index(['annee_rp', 'C22_RP_SUROCC_ACC', 'C22_RP_SUROCC_MOD', 'P22_ANEM_RP',
       'P22_APPART', 'P22_LOG', 'P22_LOGVAC', 'P22_MEN', 'P22_NPER_RP',
       'P22_RP', 'P22_RP_3040M2', 'P22_RP_ACH1919', 'P22_RP_ACH1945',
       'P22_RP_ACH2019', 'P22_RP_GARL', 'P22_RP_M30M2', 'P22_RP_VOIT1P',
       'P22_RSECOCC', 'CODGEO_INI', 'CODGEO_2026', 'BV2022', 'TYPE_BV2022',
       'sous_bassin_de_vie', 'pop_bv_totale', 'population_sbv',
       'superficie_sbv_km2', 'annee', 'n_appart', 'n_maison',
       'prix_m2_med_appart', 'prix_m2_med_maison'],
      dtype='str')


## 7. Après la fusion
- Clé : `code_commune`, sans année (recensement 2022 fixe pour toutes les années de vente).
- Effectifs additionnés à la maille de modélisation, puis calcul des 12 indicateurs avec les formules de la préparation (bloc 7) ; `log_nb_logements` = log10 du nombre de logements.
- Redondances à contrôler sur les indicateurs : taux de vacance du recensement et de LOVAC ; taille des ménages (logement) et taille moyenne des ménages (ménages).
- Transformation éventuelle et mise à l'échelle juste avant la modélisation (paramètres calculés sur les années d'apprentissage seulement).

Calcule les 12 indicateurs à partir d'effectifs additionnés à une maille (mêmes formules que le bloc 7 de la préparation)

In [32]:
def indicateurs_logement(e):
    """Calcule les 12 indicateurs à partir d'effectifs additionnés à une maille
    (mêmes formules que le bloc 7 de la préparation)."""
    e = e.copy()
    for d in ["P22_LOG", "P22_RP", "P22_MEN"]:
        e[d] = e[d].where(e[d] > 0)   # dénominateur nul -> valeur vide au lieu d'un infini

    r = pd.DataFrame(index=e.index)
    r["taux_vacance"] = e["P22_LOGVAC"] / e["P22_LOG"] * 100
    r["part_rsec"] = e["P22_RSECOCC"] / e["P22_LOG"] * 100
    r["part_apparts"] = e["P22_APPART"] / e["P22_LOG"] * 100
    r["log_nb_logements"] = np.log10(e["P22_LOG"])
    r["part_moins40m2"] = (e["P22_RP_M30M2"] + e["P22_RP_3040M2"]) / e["P22_RP"] * 100
    r["part_avant1946"] = (e["P22_RP_ACH1919"] + e["P22_RP_ACH1945"]) / e["P22_RP"] * 100
    r["part_2006_2019"] = e["P22_RP_ACH2019"] / e["P22_RP"] * 100
    r["anciennete_moyenne"] = e["P22_ANEM_RP"] / e["P22_RP"]
    r["taille_menage"] = e["P22_NPER_RP"] / e["P22_RP"]
    r["part_sans_voiture"] = 100 - e["P22_RP_VOIT1P"] / e["P22_MEN"] * 100
    r["part_parking"] = e["P22_RP_GARL"] / e["P22_MEN"] * 100
    r["part_suroccupes"] = (e["C22_RP_SUROCC_MOD"] + e["C22_RP_SUROCC_ACC"]) / e["P22_RP"] * 100
    return r

- Doublons à vérifier à la maille finale, en priorité part des moins de 40 m² / part des logements suroccupés, et part sans voiture / part avec parking.

# Feature engineering – Filosofi (niveau de vie)
**Contributeurs :** Cécile & Théophile · **Date :** 27/10/2026

Maille : commune. La cible n'est pas utilisée ici (déjà étudiée dans l'EDA Filosofi).

## 0. Chargement
Table produite par le bloc 5 de la préparation : une ligne par commune et par année, de 2014 à 2021, chaque millésime dans sa propre géographie.

In [88]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from config import DATA_OUT

df = pd.read_parquet(DATA_OUT / "filosofi_commune_2014_2021.parquet")

print(df.shape)
print("Doublons commune x année :", df.duplicated(["code_commune", "annee"]).sum())
print("Années :", sorted(df["annee"].unique()))

(261319, 29)
Doublons commune x année : 0
Années : [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]


## 1. Constats de l'EDA utilisés

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| Niveau de vie médian | C1 : lien fort avec le niveau des prix, rho = 0,66 (maisons et appartements), stable de 0,61 à 0,66 sur 2014-2021 | Gardé (la cible retenue le 26/10 est le niveau du prix) |
| Niveau de vie médian | B3 : très stable, rho = 0,90 entre 2014 et 2021 ; 88 % des écarts viennent des différences entre communes | Une photo suffit : millésime 2021 pour toutes les années de vente |
| Indicateurs détaillés (Gini, déciles, parts de revenus) | Masqués pour environ 84 % des communes ; Gini fragile ; non recalculables au territoire | Exclus |
| Valeurs masquées | En 2021, médiane masquée pour 3 604 communes sur 34 926 (10 %), surtout de très petites communes ; leur nombre de ménages fiscaux est masqué aussi | Laissées vides ; communes non comptées dans le calcul au territoire ; part des ménages concernés à contrôler avec le recensement (`P22_MEN`) |
| Paris, Lyon, Marseille | Présentes deux fois : ville entière et arrondissements | Ville entière retirée, arrondissements gardés |

## 2. Sélection des variables
Niveau de vie médian de 2021 et nombre de ménages fiscaux, qui servira de poids pour passer au territoire (on ne peut pas additionner des médianes).

In [89]:
MILLESIME = 2021   # photo : millésime le plus récent

fe = df.loc[df["annee"] == MILLESIME, ["code_commune", "niveau_vie_median", "NBMEN"]].copy()
# Paris, Lyon, Marseille : arrondissements gardés, ville entière retirée (sinon comptée deux fois)
fe = fe[~fe["code_commune"].isin(["75056", "69123", "13055"])]
fe = fe.rename(columns={"NBMEN": "nb_menages_fiscaux"})
fe["millesime_filosofi"] = MILLESIME

print(fe.shape)
print("Doublons :", fe.duplicated("code_commune").sum(),
      "| médianes masquées :", fe["niveau_vie_median"].isna().sum())
fe.describe().round(0)

(34926, 4)
Doublons : 0 | médianes masquées : 3604


,niveau_vie_median,nb_menages_fiscaux,millesime_filosofi
count,31322.0,31322.0,34926.0
mean,23244.0,915.0,2021.0
std,3174.0,4044.0,0.0
min,12260.0,35.0,2021.0
25%,21270.0,109.0,2021.0
50%,22810.0,226.0,2021.0
75%,24710.0,556.0,2021.0
max,55340.0,222023.0,2021.0


## 3. Variables qualitatives
Aucune.

## 4. Tests statistiques
Faits dans l'EDA Filosofi (B3, C1) ; à refaire au territoire après la fusion.

## 5. Variables retenues

In [91]:
print(fe.shape)

(34926, 4)


## 6. Fusion des variables retenues avec la cible, export

La fusion avec la cible doit se faire en plusieurs étapes : d'abord la jonction entre le fichier après feature engineering pour avoir pour chaque communes les informations liées au dataset travaillé et le sous bassin de vie auquel chaque commune fait partie, puis la jonction avec la table cible via le sous bassin de vie et l'année des observations.

In [92]:
cible=pd.read_parquet(DATA_OUT / "cible_sbv_annee.parquet")
liaison=pd.read_parquet(DATA_OUT / "liaison_bv_scindes.parquet")

com_fe=pd.merge(liaison, fe, left_on="CODGEO_INI", right_on="code_commune").drop(columns='code_commune')
df_final=pd.merge(com_fe,cible, left_on=["sous_bassin_de_vie"], right_on=["sous_bassin_de_vie"])
print(df_final.columns)

df_final.to_parquet(DATA_OUT / "fe_cible_filosofi_commune.parquet", index=False)

Index(['CODGEO_INI', 'CODGEO_2026', 'BV2022', 'TYPE_BV2022',
       'sous_bassin_de_vie', 'pop_bv_totale', 'population_sbv',
       'superficie_sbv_km2', 'niveau_vie_median', 'nb_menages_fiscaux',
       'millesime_filosofi', 'annee', 'n_appart', 'n_maison',
       'prix_m2_med_appart', 'prix_m2_med_maison'],
      dtype='str')


## 7. Après la fusion
- Clé : `code_commune` (géographie du millésime 2021), rattaché au territoire par la table de liaison (`CODGEO_INI`) ; sans année (photo 2021 pour toutes les années de vente).
- Niveau de vie du territoire : moyenne des médianes communales pondérée par le nombre de ménages fiscaux, sur les communes dont la médiane est connue (approximation : une moyenne de médianes n'est pas la médiane du territoire).
- Contrôle : part des ménages du territoire dans une commune à médiane masquée.
- Limite à rappeler : le millésime 2021, publié en 2024, est postérieur aux ventes d'apprentissage ; sans effet si le classement des territoires est stable (EDA, B3).

# Feature engineering – DVF+ (volume de ventes)
**Contributeurs :** Cécile & Théophile· **Date :** 27/10/2026

Maille : commune × année. La cible n'est pas utilisée ici.

## 0. Chargement
Table produite par le bloc 4 de la préparation : nombre de ventes et prix médian au m² par commune (géographie 2026, arrondissements séparés) et par année, maisons et appartements, toutes les ventes retenues (pas l'échantillon).

In [93]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from config import DATA_OUT

df = pd.read_parquet(DATA_OUT / "cible_commune_annee.parquet")

print(df.shape)
print("Doublons :", df.duplicated(["code_commune", "annee"]).sum())
print("Années :", sorted(df["annee"].unique()))

(362187, 6)
Doublons : 0
Années : [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 1. Constats utilisés

| Variable | Constat | Traitement |
|---|---|---|
| Nombre de ventes | Volet séries temporelles : le marché s'ajuste d'abord par les volumes puis par les prix (test de Granger : le volume précède le prix des maisons, p = 0,001). Seule variable du projet qui varie à la fois d'un territoire à l'autre et d'une année à l'autre | Gardé, maisons et appartements séparés |
| Nombre de ventes de l'année N | Inconnu au moment de prévoir le prix de N | Décalage d'un an : les ventes de N-1 servent pour le prix de N |
| Prix médian | C'est la cible | Exclu de ce fichier |

## 2. Sélection des variables et décalage
Nombres de ventes seulement : ils s'additionnent à n'importe quelle maille. L'évolution du volume (N-2 → N-1) sera calculée après la fusion, sur les nombres additionnés au territoire.

In [94]:
CLES = ["code_commune", "annee"]   # annee = année des ventes que ces volumes servent à expliquer

fe = df[["code_commune", "annee", "n_maison", "n_appart"]].rename(
    columns={"n_maison": "n_ventes_maison_n1", "n_appart": "n_ventes_appart_n1"})

# annee = année du prix (ventes de N-1 -> prix de N) ; l'année des ventes est gardée
fe["annee_ventes"] = df["annee"]
fe["annee"] = df["annee"] + 1

print(fe.shape)
print("Doublons sur les clés :", fe.duplicated(CLES).sum())
print(fe[["annee_ventes", "annee"]].drop_duplicates().sort_values("annee").to_string(index=False))

(362187, 5)
Doublons sur les clés : 0
 annee_ventes  annee
         2014   2015
         2015   2016
         2016   2017
         2017   2018
         2018   2019
         2019   2020
         2020   2021
         2021   2022
         2022   2023
         2023   2024
         2024   2025
         2025   2026


## 3. Variables qualitatives
Aucune.

## 4. Tests statistiques
Reportés après la fusion : ils doivent porter sur les volumes additionnés au territoire et sur leur évolution.

## 5. Variables retenues et export

In [95]:
VARIABLES = ["annee_ventes", "n_ventes_maison_n1", "n_ventes_appart_n1"]

df_fe = fe[CLES + VARIABLES]

print(df_fe.shape)
print("Doublons sur les clés :", df_fe.duplicated(CLES).sum())

df_fe.to_parquet(DATA_OUT / "fe_dvf_volume_commune.parquet", index=False)

(362187, 5)
Doublons sur les clés : 0


## 6. Après la fusion
- Nombres de ventes additionnés au territoire (communes rattachées par la table de liaison ; `code_commune` est en géographie 2026, arrondissements séparés).
- Évolution du volume entre N-2 et N-1, par type de bien : (ventes N-1 − ventes N-2) / ventes N-2, à partir des nombres additionnés.
- Une commune sans vente une année est absente de la table du bloc 4 : son volume vaut 0 pour cette année, à compléter après la jointure.
- Ventes de 2025 (prix 2026) : hors apprentissage et test, réservées à la simulation finale.

# Feature engineering – Taux des crédits immobiliers
**Contributeurs :** Cécile & Théophile· **Date :** 28/10/2026

Maille : France entière, une valeur par mois (Banque de France). Table produite par le bloc 11 de la préparation (mois, trimestre, année).

## 1. Constats utilisés

| Constat | Source | Conséquence |
|---|---|---|
| Le taux est le même pour tous les territoires un mois donné | Bloc 11 | Dans une table territoire × année, il ne prend qu'une valeur par année |
| De 2020 à 2023, le modèle ne voit que 4 valeurs du taux, toutes basses | Baseline | Il ne peut pas séparer l'effet du taux de tout ce qui s'est passé la même année |
| En 2024 et 2025, le taux sort de la plage vue en apprentissage | Baseline | Une régression prolonge sa droite, un arbre répète la dernière valeur vue : la prévision n'est pas fiable |
| Ajouté à la baseline, le taux décale toutes les prévisions de 2024 d'environ 890 €/m² | Baseline (SHAP) | Il dégrade la prévision au lieu de l'améliorer |
| À la maille mensuelle, le taux apporte peu à la prévision du prix | Volet séries temporelles (SARIMAX) | Même avec une valeur par mois, son apport reste faible |

## 2. Décision
Le taux des crédits n'entre pas dans la modélisation annuelle par territoire. Il est étudié seulement dans le volet séries temporelles (SARIMAX), à la maille mensuelle, où il prend une valeur différente chaque mois.

Aucun fichier exporté.